# Breast Tumor Classification using ANN

## Problem
Predict whether a tumor is **Benign (B)** or **Malignant (M)**.

## ANN architecture
- Input: 30 original numerical features
- Feature engineering: 4 ratio features
- StandardScaler
- Hidden layer 1: 64 neurons, ReLU
- Hidden layer 2: 32 neurons, ReLU
- Output: 1 probability/class
- Optimizer: Adam
- Early stopping: enabled

> Educational project only; not a medical diagnostic system.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
from sklearn.base import BaseEstimator, TransformerMixin
import joblib

In [ ]:
df=pd.read_csv("data.csv")
print("Shape:",df.shape)
display(df.head())
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).head())
print("Duplicate rows:",df.duplicated().sum())

In [ ]:
df=df.drop(columns=["Unnamed: 32","id"])
df["diagnosis"]=df["diagnosis"].map({"B":0,"M":1})
display(df.describe().T)

## EDA

In [ ]:
sns.countplot(data=df,x="diagnosis")
plt.xticks([0,1],["Benign","Malignant"])
plt.title("Target Distribution")
plt.show()

df.hist(figsize=(18,16),bins=25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18,14))
sns.heatmap(df.corr(numeric_only=True),cmap="coolwarm",center=0)
plt.title("Correlation Heatmap")
plt.show()

## Feature engineering and split

In [ ]:
class FeatureEngineer(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): return self
    def transform(self,X):
        X=X.copy(); e=1e-9
        X["area_radius_ratio"]=X["area_mean"]/(X["radius_mean"]+e)
        X["perimeter_radius_ratio"]=X["perimeter_mean"]/(X["radius_mean"]+e)
        X["worst_mean_radius_ratio"]=X["radius_worst"]/(X["radius_mean"]+e)
        X["worst_mean_area_ratio"]=X["area_worst"]/(X["area_mean"]+e)
        return X

X=df.drop(columns="diagnosis")
y=df["diagnosis"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
print("Train:",X_train.shape,"Test:",X_test.shape)

## Build and train the ANN
This is an actual Artificial Neural Network implemented as a scikit-learn Multi-Layer Perceptron (`MLPClassifier`).

In [ ]:
ann=Pipeline([
    ("feature_engineering",FeatureEngineer()),
    ("scaler",StandardScaler()),
    ("ann",MLPClassifier(
        hidden_layer_sizes=(64,32),
        activation="relu",
        solver="adam",
        alpha=0.0005,
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        random_state=42
    ))
])

ann.fit(X_train,y_train)
pred=ann.predict(X_test)
prob=ann.predict_proba(X_test)[:,1]

print("Accuracy :",accuracy_score(y_test,pred))
print("Precision:",precision_score(y_test,pred))
print("Recall   :",recall_score(y_test,pred))
print("F1       :",f1_score(y_test,pred))
print("ROC-AUC  :",roc_auc_score(y_test,prob))
print("\n",classification_report(y_test,pred,target_names=["Benign","Malignant"]))

In [ ]:
cm=confusion_matrix(y_test,pred)
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Benign","Malignant"],yticklabels=["Benign","Malignant"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("ANN Confusion Matrix"); plt.show()

RocCurveDisplay.from_predictions(y_test,prob)
plt.title("ANN ROC Curve"); plt.show()

In [ ]:
# Save the trained ANN pipeline
joblib.dump(ann,"ann_model.joblib")
print("Saved ann_model.joblib")

## Streamlit deployment

Run:

```bash
pip install -r ann_requirements.txt
streamlit run ann_app.py
```

The app provides manual prediction and prediction of a selected row from the dataset.